# Data Cleaning for House Price Classification

**Project goal:** prepare the raw King County housing dataset so it can be used to predict the price category of a house, including an `overpriced` class and four additional price classes.
**Dataset:** Kaggle, *House Sales in King County, USA*

This notebook is intentionally focused on data preparation only. It removes invalid records, standardizes labels, creates useful engineered features, and exports a clean dataset that is ready for the modeling stage.


## 1. Setup and configuration

This cell imports the libraries used for cleaning, sets the random seed for reproducibility, and defines the file paths for the raw source data and the cleaned outputs.


In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import (
    add_model_features,
    add_renovated_flag,
    drop_duplicates,
    drop_metadata_columns,
    fix_types,
    remove_impossible_rows,
)
from src.validation import validate_cleaned_data

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RAW_PATH = '../data/raw/usa_house_data.csv'
CLEAN_PATH = '../data/processed/house_clean.csv'
SAMPLE_PATH = '../sample_data/sample_houses.csv'

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
warnings.filterwarnings('ignore')

os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
os.makedirs(os.path.dirname(SAMPLE_PATH), exist_ok=True)

print('Setup complete. Random state =', RANDOM_STATE)


Setup complete. Random state = 42


## 2. Load the raw dataset

This cell reads the original CSV file and confirms the dataset dimensions before any cleaning steps begin.


In [2]:
df = pd.read_csv(RAW_PATH)
print(f'Loaded raw data from: {RAW_PATH}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())


Loaded raw data from: ../data/raw/usa_house_data.csv
Shape: 21,613 rows x 21 columns


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,"221,900.00",3,1.00,1180,5650,1.00,0,0,3,7,1180,0,1955,0,98178,47.51,-122.26,1340,5650
1,6414100192,20141209T000000,"538,000.00",3,2.25,2570,7242,2.00,0,0,3,7,2170,400,1951,1991,98125,47.72,-122.32,1690,7639
2,5631500400,20150225T000000,"180,000.00",2,1.00,770,10000,1.00,0,0,3,6,770,0,1933,0,98028,47.74,-122.23,2720,8062
3,2487200875,20141209T000000,"604,000.00",4,3.00,1960,5000,1.00,0,0,5,7,1050,910,1965,0,98136,47.52,-122.39,1360,5000
4,1954400510,20150218T000000,"510,000.00",3,2.00,1680,8080,1.00,0,0,3,8,1680,0,1987,0,98074,47.62,-122.05,1800,7503


## 3. Inspect schema and missing values

This cell checks the column types and reports missing values so we can decide whether the raw data requires type fixes or data removal before modeling.


In [3]:
print('=== Data types ===')
print(df.dtypes.to_string())

missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print('\n=== Missing values ===')
display(missing)


=== Data types ===
id                 int64
date                 str
price            float64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64

=== Missing values ===


,missing_count,missing_pct
id,0,0.00
date,0,0.00
price,0,0.00
bedrooms,0,0.00
bathrooms,0,0.00
sqft_living,0,0.00
sqft_lot,0,0.00
floors,0,0.00
waterfront,0,0.00
view,0,0.00


## 4. Fix data types and validate categorical ranges

This cell converts dates to datetime values, stores ZIP codes as string labels, and confirms that ordinal columns remain within their valid ranges. These checks prevent malformed values from propagating into the model.


In [4]:
df = fix_types(df)

range_checks = {
    'waterfront': (0, 1),
    'view': (0, 4),
    'condition': (1, 5),
    'grade': (1, 13),
}

for col, (low, high) in range_checks.items():
    bad = df[(df[col] < low) | (df[col] > high)]
    print(f'{col}: {len(bad)} rows outside [{low}, {high}]')

print(f"date dtype: {df['date'].dtype}")
print(f"zipcode dtype: {df['zipcode'].dtype}")


waterfront: 0 rows outside [0, 1]
view: 0 rows outside [0, 4]
condition: 0 rows outside [1, 5]
grade: 0 rows outside [1, 13]
date dtype: datetime64[us]
zipcode dtype: str


## 5. Remove duplicate rows and repeated property records

This cell removes fully duplicated rows and retains only the most recent record for any house that appears more than once. Repeated sales would otherwise distort the learning task and create leakage between training examples.


In [5]:
rows_before = len(df)
df = drop_duplicates(df)

print(f'Rows before duplicate handling: {rows_before:,}')
print(f'Rows after duplicate handling: {len(df):,}')
print(f'Removed duplicate rows: {rows_before - len(df):,}')


Rows before duplicate handling: 21,613
Rows after duplicate handling: 21,436
Removed duplicate rows: 177


## 6. Remove impossible and clearly invalid records

This cell removes rows that cannot represent a valid sale, such as negative or zero prices, zero living space, and impossible bedroom counts. These observations are not useful for training and would bias the model.


In [6]:
rows_before = len(df)
df = remove_impossible_rows(df)

print(f'Rows before invalid-record removal: {rows_before:,}')
print(f'Rows after invalid-record removal: {len(df):,}')
print(f'Removed invalid rows: {rows_before - len(df):,}')


Rows before invalid-record removal: 21,436
Rows after invalid-record removal: 21,428
Removed invalid rows: 8


## 7. Build model-ready derived features

This cell creates a few compact features that improve interpretability and help the final classifier without carrying unnecessary raw metadata. The aim is to keep the cleaned dataset practical for prediction while removing information that users would not normally provide.


In [7]:
df = add_model_features(df)
df = add_renovated_flag(df)
df = drop_metadata_columns(df)

print('Derived columns added: sale_year, price_per_sqft, renovated')
print('Remaining columns:')
print(df.columns.tolist())


Derived columns added: sale_year, price_per_sqft, renovated
Remaining columns:
['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'zipcode', 'sale_year', 'price_per_sqft', 'renovated']


## 8. Final validation before export

This cell performs a final sanity check on the cleaned dataset and confirms that the features are ready to be exported for classification. It validates the core data assumptions before the file is saved.


In [8]:
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('\nFinal dtypes:')
print(df.dtypes.to_string())

validation_errors = validate_cleaned_data(df)
assert not validation_errors, '; '.join(validation_errors)
print('Validation checks passed.')


Final shape: 21,428 rows x 17 columns

Final dtypes:
price             float64
bedrooms            int64
bathrooms         float64
sqft_living         int64
sqft_lot            int64
floors            float64
waterfront          int64
view                int64
condition           int64
grade               int64
sqft_above          int64
sqft_basement       int64
yr_built            int64
zipcode               str
sale_year           int32
price_per_sqft    float64
renovated           int64


Validation checks passed.


## 9. Save the cleaned dataset and a sample for testing

This cell saves the cleaned data to the project data folder and exports a 30-row sample so the application and modeling pipeline can be tested quickly without loading the full dataset each time.


In [9]:
df['zipcode'] = df['zipcode'].astype('string')
df.to_csv(CLEAN_PATH, index=False)

sample_df = df.sample(n=min(30, len(df)), random_state=RANDOM_STATE)
sample_df.to_csv(SAMPLE_PATH, index=False)

print(f'Clean dataset saved to: {CLEAN_PATH}')
print(f'Sample dataset saved to: {SAMPLE_PATH}')
print(f'Sample shape: {sample_df.shape}')
display(sample_df.head())


Clean dataset saved to: ../data/processed/house_clean.csv
Sample dataset saved to: ../sample_data/sample_houses.csv
Sample shape: (30, 17)


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,zipcode,sale_year,price_per_sqft,renovated
15324,"402,000.00",4,1.75,2430,5481,1.00,0,0,3,8,1430,1000,1953,98108,2014,165.43,0
6076,"425,000.00",2,1.00,1160,5038,1.00,0,0,5,7,740,420,1942,98126,2015,366.38,0
2534,"699,000.00",4,2.50,3230,40319,2.00,0,0,4,10,3230,0,1987,98027,2015,216.41,0
14526,"248,500.00",2,1.00,720,5160,1.00,0,0,3,6,720,0,1949,98106,2014,345.14,0
4393,"649,000.00",2,1.00,1280,4840,1.00,0,0,3,7,1200,80,1908,98115,2015,507.03,0


## Summary

The dataset is now cleaned and standardized for multiclass house price classification. The final output contains realistic records, fixed field types, and a reduced set of model-ready features that are appropriate for predictive use.
